# Task 03 — Text Generation with Markov Chains

**GenAI Internship — Task 03**

Goal: implement a simple text generation algorithm using Markov chains — a statistical model that
predicts the probability of the next character or word based on the previous n character(s)/word(s) —
and use it to generate new, statistically-plausible text from a source corpus.

**Pipeline**
1. Load a custom text dataset (a public-domain short story collection)
2. Build a **character-level** Markov chain from scratch (n-gram table + random weighted choice)
3. Build a **word-level** Markov chain from scratch
4. Generate text from both, comparing different chain **orders** (n-gram sizes)
5. Use the `markovify` library for more robust, sentence-aware generation
6. Compare all approaches side-by-side and discuss trade-offs

> This notebook has no GPU requirement — everything here runs comfortably on CPU, including in Colab's
> default runtime.


## 1. Load a custom text dataset

We'll use **"The Wonderful Wizard of Oz"** by L. Frank Baum — a short, public-domain novel from
[Project Gutenberg](https://www.gutenberg.org/) — as our training corpus. Its simple, narrative prose
style makes the Markov chain's output easy to read and evaluate.

To use your **own** text instead, replace the download cell with any plain `.txt` file of your choice —
Markov chains work on any body of text.


In [ ]:
import urllib.request

GUTENBERG_URL = "https://www.gutenberg.org/files/55/55-0.txt"

raw_text = urllib.request.urlopen(GUTENBERG_URL).read().decode("utf-8-sig")

# Project Gutenberg files wrap the actual book between standard boilerplate markers —
# strip the legal header/footer so the Markov model only trains on the story itself.
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_idx = raw_text.find(start_marker)
start_idx = raw_text.find("\n", start_idx) + 1  # skip past the marker line itself
end_idx = raw_text.find(end_marker)

text = raw_text[start_idx:end_idx].strip()
text = text.replace("\r\n", "\n")

print(f"Corpus length: {len(text):,} characters")
print(text[:600])


## 2. Character-level Markov chain (built from scratch)

**How it works:**
1. Slide a window of size `order` across the text, recording each n-gram (sequence of `order`
   characters) and the character that immediately follows it.
2. This builds a table: `n-gram -> [list of characters that followed it in the corpus]`.
3. To generate: start with a real n-gram from the text, then repeatedly look up the last `order`
   characters of the output in the table and append a **randomly chosen** character from the matching
   list. Repeat.

Higher `order` means the model has "seen" a more specific context, so its output looks more like the
original text; lower `order` means more randomness and novelty (but more nonsense).


In [ ]:
import random
from collections import defaultdict

def build_char_markov_table(text, order=4):
    table = defaultdict(list)
    for i in range(len(text) - order):
        gram = text[i:i + order]
        next_char = text[i + order]
        table[gram].append(next_char)
    return table

def generate_char_text(table, order, length=500, seed=None):
    if seed is None:
        seed = random.choice(list(table.keys()))
    output = seed
    for _ in range(length):
        gram = output[-order:]
        possible_next = table.get(gram)
        if not possible_next:
            # dead end — restart from a fresh random n-gram seen in training
            gram = random.choice(list(table.keys()))
            output += gram
            continue
        output += random.choice(possible_next)
    return output

char_table_order4 = build_char_markov_table(text, order=4)
generated_char_o4 = generate_char_text(char_table_order4, order=4, length=500)
print(generated_char_o4)


### Effect of order on character-level generation

Let's compare a low order (more random/novel, less coherent) against a high order (closer to copying
the source, but grammatically safer).


In [ ]:
for order in [2, 4, 8]:
    table = build_char_markov_table(text, order=order)
    generated = generate_char_text(table, order=order, length=300)
    print(f"--- order={order} ---")
    print(generated)
    print()


## 3. Word-level Markov chain (built from scratch)

Same idea as the character-level model, but the units are **words** instead of characters. This tends to
produce output that's grammatically looser at the sentence level but uses only real whole words — a
different, often more "readable" trade-off than the character-level model.


In [ ]:
import re

def tokenize_words(text):
    # Simple whitespace + punctuation-aware tokenizer
    return re.findall(r"[A-Za-z']+|[.,!?;]", text)

words = tokenize_words(text)
print(f"Total word tokens: {len(words):,}")

def build_word_markov_table(words, order=2):
    table = defaultdict(list)
    for i in range(len(words) - order):
        gram = tuple(words[i:i + order])
        next_word = words[i + order]
        table[gram].append(next_word)
    return table

def generate_word_text(table, order, num_words=100, seed=None):
    if seed is None:
        seed = random.choice(list(table.keys()))
    output = list(seed)
    for _ in range(num_words):
        gram = tuple(output[-order:])
        possible_next = table.get(gram)
        if not possible_next:
            gram = random.choice(list(table.keys()))
            output.extend(gram)
            continue
        output.append(random.choice(possible_next))

    # Join words back into readable text (naive punctuation spacing)
    result = ""
    for tok in output:
        if tok in ".,!?;":
            result = result.rstrip() + tok + " "
        else:
            result += tok + " "
    return result.strip()

word_table_order2 = build_word_markov_table(words, order=2)
generated_word_o2 = generate_word_text(word_table_order2, order=2, num_words=100)
print(generated_word_o2)


In [ ]:
for order in [1, 2, 3]:
    table = build_word_markov_table(words, order=order)
    generated = generate_word_text(table, order=order, num_words=80)
    print(f"--- word-level order={order} ---")
    print(generated)
    print()


## 4. Using `markovify` for sentence-aware generation

Our from-scratch models generate a fixed-length blob of text without any notion of "sentence." The
[`markovify`](https://github.com/jsvine/markovify) library builds a similar n-gram model but is
**sentence-aware**: it splits the corpus into sentences first, then guarantees that generated output
starts and ends like a grammatically plausible sentence. This is the more practical, production-friendly
way to do Markov-chain text generation.


In [ ]:
!pip install -q markovify


In [ ]:
import markovify

# state_size is markovify's name for the Markov chain "order"
markov_model = markovify.Text(text, state_size=2)

print("Randomly generated sentences:\n")
for _ in range(5):
    sentence = markov_model.make_sentence(tries=100)
    if sentence:
        print("-", sentence)


In [ ]:
# Constrain output length, or start with specific text
print("Short sentences (max 100 characters):\n")
for _ in range(5):
    sentence = markov_model.make_short_sentence(max_chars=100, tries=100)
    if sentence:
        print("-", sentence)


### Comparing `markovify` state sizes (order)

Just like our from-scratch model, increasing `state_size` makes markovify's output closer to the
original text (safer grammar, less novelty); decreasing it makes output more creative but less coherent.


In [ ]:
for state_size in [1, 2, 3]:
    model = markovify.Text(text, state_size=state_size)
    print(f"--- markovify state_size={state_size} ---")
    for _ in range(3):
        sentence = model.make_sentence(tries=100)
        print("-", sentence)
    print()


## 5. Try your own text / prompt

Swap in any text of your own (fairy tales, song lyrics you own the rights to, product reviews, etc.) by
replacing `text` above, or generate more sentences below using the existing model.


In [ ]:
num_sentences = 8  #@param {type:"integer"}
state_size = 2  #@param {type:"integer"}

your_model = markovify.Text(text, state_size=state_size)
for _ in range(num_sentences):
    sentence = your_model.make_sentence(tries=100)
    if sentence:
        print("-", sentence)


## 6. Save generated samples to disk

In [ ]:
import os
import json as json_lib

OUTPUT_DIR = "generated_text"
os.makedirs(OUTPUT_DIR, exist_ok=True)

samples = {
    "char_level_order4": generate_char_text(build_char_markov_table(text, order=4), order=4, length=400),
    "word_level_order2": generate_word_text(build_word_markov_table(words, order=2), order=2, num_words=100),
    "markovify_sentences": [markov_model.make_sentence(tries=100) for _ in range(10)],
}

with open(os.path.join(OUTPUT_DIR, "samples.json"), "w") as f:
    json_lib.dump(samples, f, indent=2)

print("Saved to", os.path.join(OUTPUT_DIR, "samples.json"))
for key, value in samples.items():
    print(f"\n--- {key} ---")
    print(value)


## Conclusion

- Built **character-level** and **word-level** Markov chains from scratch, using n-gram frequency tables
  and weighted-random next-token selection.
- Showed how the chain's **order** trades off novelty (low order) against faithfulness to the source
  (high order), at both the character and word level.
- Used **`markovify`** for a more practical, sentence-aware version of the same idea.
- Saved sample generations for inclusion in the GitHub repo.

**Next steps / extensions:** try a much larger corpus for richer vocabulary, combine markovify's sentence
awareness with a higher-order character model, or compare this statistical approach's coherence against
the GPT-2 model fine-tuned in Task 01.
